# Package Installation

Open this notebook in **Google Colab** (or locally), run this cell first, then the UI cells below.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vincenzoos/IPSAE-notebook/blob/main/ipsae_eval.ipynb)

**Colab tips**
- Uploads: use the Single Model / Bulk Evaluation upload widgets (zip for bulk folders).
- Downloads: use the zip helper cell at the bottom, then download from the Colab file browser.
- Runtime: CPU is enough for ipSAE scoring.


In [ ]:
# Setup for local Jupyter or Google Colab
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output as colab_output

    colab_output.enable_custom_widget_manager()
    REPO_URL = "https://github.com/Vincenzoos/IPSAE-notebook.git"
    REPO_DIR = Path("/content/IPSAE-notebook")
    if not (REPO_DIR / "ipsae.py").exists():
        !git clone --depth 1 {REPO_URL} "{REPO_DIR}"
    %cd /content/IPSAE-notebook

%pip install -q numpy 'ipywidgets>=8' jupyterlab_widgets widgetsnbextension
import ipywidgets

print(f"ipywidgets {ipywidgets.__version__} OK")
print(f"cwd: {Path.cwd().resolve()}")


# ipSAE Evaluation

Run single-model or bulk AlphaFold Server evaluations. Upload structure/PAE files (or a zip for bulk), or type paths already on the machine.


In [ ]:
import sys
from pathlib import Path

def _find_additions(marker: str) -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (
            base / "additions",
            base / "dependencies" / "IPSAE" / "additions",
        ):
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        f"Could not find additions/{marker}. "
        "If you are on Colab, run the Package Installation cell first."
    )

ADDITIONS = _find_additions("ipsae_eval_ui.py")
if str(ADDITIONS) not in sys.path:
    sys.path.insert(0, str(ADDITIONS))

from ipsae_eval_ui import launch_ipsae_eval_ui

launch_ipsae_eval_ui()


# ipSAE Comparison CSV

Summarize collected ipSAE outputs (for example a `bulk_ipsae_evals_YYYYMMDD_HHMMSS/` folder) into a ranked comparison table / CSV.


In [ ]:
import importlib
import sys
from pathlib import Path

def _find_additions(marker: str) -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (
            base / "additions",
            base / "dependencies" / "IPSAE" / "additions",
        ):
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        f"Could not find additions/{marker}. "
        "If you are on Colab, run the Package Installation cell first."
    )

ADDITIONS = _find_additions("ipsae_comparison_ui.py")
if str(ADDITIONS) not in sys.path:
    sys.path.insert(0, str(ADDITIONS))

import ipsae_comparison_ui

importlib.reload(ipsae_comparison_ui)
ipsae_comparison_ui.launch_ipsae_comparison_ui()


In [ ]:
# Zip a folder for download (useful on Colab / remote notebooks)
import shutil
import subprocess
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

def _find_additions(marker: str) -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (
            base / "additions",
            base / "dependencies" / "IPSAE" / "additions",
        ):
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        f"Could not find additions/{marker}. "
        "If you are on Colab, run the Package Installation cell first."
    )

ADDITIONS = _find_additions("folder_picker.py")
if str(ADDITIONS) not in sys.path:
    sys.path.insert(0, str(ADDITIONS))

from folder_picker import folder_value, make_folder_picker
from paths import ROOT as REPO_ROOT

folder_dd, refresh_btn, folder_row = make_folder_picker(description="Folder", dropdown_width="520px")
zip_btn = widgets.Button(description="Zip folder", button_style="success", icon="file-archive")
status = widgets.HTML(value=f'<span style="color:#57606a">Select a folder under {REPO_ROOT}, then Zip.</span>')
log = widgets.Output()


def _zip_folder(_):
    name = folder_value(folder_dd)
    src = REPO_ROOT / name
    with log:
        clear_output(wait=True)
        if not name or not src.is_dir():
            status.value = '<span style="color:#cf222e">Folder not found.</span>'
            return
        if shutil.which("zip") is None:
            subprocess.run(["apt-get", "update"], check=False)
            subprocess.run(["apt-get", "install", "-y", "zip", "unzip"], check=True)
        print(f"zip -r {name}.zip {name}")
        result = subprocess.run(["zip", "-r", f"{name}.zip", name], cwd=str(REPO_ROOT))
        if result.returncode == 0:
            status.value = f'<span style="color:#1a7f37;font-weight:600">Created {REPO_ROOT / (name + ".zip")}</span>'
        else:
            status.value = '<span style="color:#cf222e">zip failed (see log).</span>'


zip_btn.on_click(_zip_folder)
display(widgets.VBox([folder_row, zip_btn, status, log]))
